# Краткое описание
- Цель эксперимента - первичный EDA и проверка качества разметки токсичных комментариев.
- Данные - toxic-russian-comments (dataset.txt из Kaggle).
- Основные выводы - распределения классов и длины текстов, обнаружены дубликаты и конфликты разметки.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

sns.set(style="whitegrid")

In [ ]:
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

data_dir = Path("../../data")
data_dir.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()
api.dataset_download_files("alexandersemiletov/toxic-russian-comments", path=str(data_dir), unzip=True)

In [ ]:
data_list = []
with open("../../data/dataset.txt", encoding="utf-8") as file:
    for line in file:
        labels = line.split()[0]
        text = line[len(labels) + 1:].strip()
        labels = labels.split(",")
        mask = [
            1 if "__label__NORMAL" in labels else 0,
            1 if "__label__INSULT" in labels else 0,
            1 if "__label__THREAT" in labels else 0,
            1 if "__label__OBSCENITY" in labels else 0,
        ]
        data_list.append((text, *mask))

In [ ]:
df = pd.DataFrame(data_list, columns=["text", "normal", "insult", "threat", "obscenity"])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe(include='all')

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Duplicate texts:", df["text"].duplicated().sum())


duplicated_texts = df[df.duplicated(subset=['text'], keep=False)]


inconsistent = []
for text, group in duplicated_texts.groupby('text'):

    if (group['normal'].nunique() > 1 or
        group['insult'].nunique() > 1 or
        group['threat'].nunique() > 1 or
        group['obscenity'].nunique() > 1):
        inconsistent.append(group)


result = pd.concat(inconsistent).sort_values(['text', 'normal', 'insult', 'threat', 'obscenity'])

print("Duplicated texts with different labels:")
display(result)

In [ ]:
print("Duplicate texts with same labels:", df.duplicated(subset=["text", "normal", "insult", "threat", "obscenity"]).sum())

Распределение классов

In [ ]:
label_cols = ["normal", "insult", "threat", "obscenity"]

label_counts = df[label_cols].sum().sort_values(ascending=False)
label_counts

In [ ]:

from pathlib import Path


def _find_project_dir(name="project", max_levels=10):
    p = Path.cwd()
    for _ in range(max_levels):
        if p.name == name:
            return p
        p = p.parent
    for ancestor in Path.cwd().parents:
        if ancestor.name == name:
            return ancestor
    return Path.cwd()


project_dir = _find_project_dir()
output_dir = project_dir / "artifacts" / "EDA" / "spans_dataset"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=label_counts.index, y=label_counts.values, palette="viridis")
plt.title("Label distribution")
plt.xlabel("")
plt.ylabel("Count")
plt.savefig(str(output_dir / "label_distribution.png"))
plt.show()

Доля каждого класса

In [ ]:
label_share = (df[label_cols].sum() / len(df) * 100).sort_values(ascending=False)
label_share

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=label_share.index, y=label_share.values, palette="magma")
plt.title("Label share, %")
plt.xlabel("")
plt.ylabel("Percent")
plt.savefig(str(output_dir / "label_share_percent.png"))
plt.show()

Количество меток на один текст

In [ ]:
df["label_sum"] = df[label_cols].sum(axis=1)
df["label_sum"].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x=df["label_sum"], color="steelblue")
plt.title("Number of labels per text")
plt.xlabel("Labels per sample")
plt.ylabel("Count")
plt.savefig(str(output_dir / "labels_per_sample_count.png"))
plt.show()

Сочетания меток

In [ ]:
def get_label_combination(row):
    labels = [col for col in label_cols if row[col] == 1]
    return ", ".join(labels) if labels else "none"

df["label_combo"] = df.apply(get_label_combination, axis=1)

In [ ]:
combo_counts = df["label_combo"].value_counts()
combo_counts.head(20)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(y=combo_counts.head(15).index, x=combo_counts.head(15).values, palette="crest")
plt.title("Top label combinations")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_label_combinations.png"))
plt.show()

Тексты и длина

In [ ]:
df["text_len_chars"] = df["text"].astype(str).str.len()
df["text_len_words"] = df["text"].astype(str).str.split().apply(len)

In [ ]:
df[["text_len_chars", "text_len_words"]].describe()

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["text_len_words"], bins=50, kde=True, color="steelblue")
plt.title("Distribution of text length in words")
plt.xlabel("Words")
plt.ylabel("Count")
plt.savefig(str(output_dir / "text_length_words_distribution.png"))
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["text_len_chars"], bins=50, kde=True, color="darkorange")
plt.title("Distribution of text length in characters")
plt.xlabel("Characters")
plt.ylabel("Count")
plt.savefig(str(output_dir / "text_length_chars_distribution.png"))
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(y=df["text_len_words"], color="lightblue")
plt.title("Text length boxplot")
plt.ylabel("Words")
plt.savefig(str(output_dir / "text_length_boxplot.png"))
plt.show()

Длина текста по меткам

In [ ]:
length_by_label = []
for label in label_cols:
    tmp = df[df[label] == 1][["text_len_words"]].copy()
    tmp["label"] = label
    length_by_label.append(tmp)

length_by_label = pd.concat(length_by_label, ignore_index=True)
length_by_label.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=length_by_label, x="label", y="text_len_words")
plt.title("Text length by label")
plt.xlabel("")
plt.ylabel("Words")
plt.savefig(str(output_dir / "text_length_by_label_boxplot.png"))
plt.show()

Частые слова

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^а-яёa-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df["text_clean"] = df["text"].apply(preprocess_text)

In [ ]:
all_words = " ".join(df["text_clean"]).split()
word_counter = Counter(all_words)

top_words = pd.DataFrame(word_counter.most_common(20), columns=["word", "count"])
top_words

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_words, y="word", x="count", color="steelblue")
plt.title("Top 20 words")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_words.png"))
plt.show()

Частые биграммы

In [ ]:
def get_top_ngrams(text_series, n=2, top_k=20):
    vectorizer = CountVectorizer(ngram_range=(n, n))
    X = vectorizer.fit_transform(text_series)
    freqs = X.sum(axis=0).A1
    ngrams = vectorizer.get_feature_names_out()
    result = pd.DataFrame({"ngram": ngrams, "count": freqs})
    return result.sort_values("count", ascending=False).head(top_k)

In [ ]:
top_bigrams = get_top_ngrams(df["text_clean"], n=2, top_k=20)
top_bigrams

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_bigrams, y="ngram", x="count", color="tomato")
plt.title("Top 20 bigrams")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_bigrams.png"))
plt.show()
